In [1]:
import pandas as pd
import glob
image_sheet = pd.read_csv("/projects/bodymaps/Data/metadata_ucsf_batch_1_to_6_and_merlin.csv")

/local/rpinise1/3997157/ipykernel_534435/4146185415.py:3: DtypeWarning: Columns (108,109,110,112,118,122,123,124,125,127,128,130,131,133,134,136,137,139,140,142,143,145,146,148,149,151,152,154,155,157,158,160,161,163,164,166,167,169,170,172,173,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266) have mixed types. Specify dtype option on import or set low_memory=False.
  image_sheet = pd.read_csv("/projects/bodymaps/Data/metadata_ucsf_batch_1_to_6_and_merlin.csv")


In [2]:
tumor_sheet = pd.read_csv("/projects/bodymaps/Data/metadata_per_tumor_ucsf_batch_1_to_6_and_merlin.csv")

/local/rpinise1/3997157/ipykernel_534435/2023293037.py:1: DtypeWarning: Columns (10,12,13,14,16,17,18,19,20,21,22,25,31,32,34,35,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,95,96,97,98,99,100,101,102) have mixed types. Specify dtype option on import or set low_memory=False.
  tumor_sheet = pd.read_csv("/projects/bodymaps/Data/metadata_per_tumor_ucsf_batch_1_to_6_and_merlin.csv")


In [3]:
tumor_sheet["Organ"].unique()

NameError: name 'tumor_sheet' is not defined

In [3]:
search_path = (
    "/projects/bodymaps/Data/"
    "radiologist_annotations_merlin_ucsf_atlas_multi_cancer/"
    "*/segmentations/*"
)
paths_with_tumor_mask = glob.glob(search_path)

In [4]:
image_sheet = image_sheet[~image_sheet["Exam Completed Date"].isna()] # get just the rows where there IS an exam completed day

In [5]:
image_sheet.shape

(124588, 267)

In [6]:
all_p_ids = image_sheet["Patient ID"]

In [7]:
all_longitudinal_ids = all_p_ids.value_counts().loc[lambda x: x > 1].index.tolist() # get all patient ids with more than one occurence (meaning multiple CT scans)

In [8]:
num_cts_per_patient_all_ids = all_p_ids[all_p_ids.isin(all_longitudinal_ids)].value_counts().to_dict() # get the number of CTs for each individual with multiple CTs

In [9]:
longitudinal_image_sheet = image_sheet[image_sheet["Patient ID"].isin(all_longitudinal_ids)]

In [10]:
longitudinal_image_sheet["datetime"] = pd.to_datetime(
    longitudinal_image_sheet["Exam Completed Date"],
    format="ISO8601",
    errors="raise"
)

/local/rpinise1/3997157/ipykernel_534435/645535468.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  longitudinal_image_sheet["datetime"] = pd.to_datetime(


In [11]:
longitudinal_image_sheet["unix timestamp"] = longitudinal_image_sheet["datetime"].astype("int64") // 10**9

/local/rpinise1/3997157/ipykernel_534435/3147996239.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  longitudinal_image_sheet["unix timestamp"] = longitudinal_image_sheet["datetime"].astype("int64") // 10**9


In [12]:
patient_to_bdmap = (
    longitudinal_image_sheet
    .drop_duplicates(subset=["Patient ID", "unix timestamp"])
    .groupby("Patient ID")["BDMAP ID"]
    .apply(list)
    .to_dict()
)

In [13]:
patient_ids = list(patient_to_bdmap.keys())

In [14]:
from itertools import permutations
import pandas as pd
from dateutil.relativedelta import relativedelta

# -----------------------
# Build lookups
# -----------------------

bdmap_to_contrast = (
    longitudinal_image_sheet
    .drop_duplicates(subset=["BDMAP ID"])
    .set_index("BDMAP ID")["contrast"]
    .to_dict()
)

# keep UNIX timestamps (no conversion stored)
bdmap_to_time = (
    longitudinal_image_sheet
    .drop_duplicates(subset=["BDMAP ID"])
    .set_index("BDMAP ID")["unix timestamp"]
    .to_dict()
)

# separate datetime lookup ONLY for delta computation
bdmap_to_datetime = {
    k: pd.to_datetime(v, unit="s")
    for k, v in bdmap_to_time.items()
}

# -----------------------
# Helper: nice timedelta
# -----------------------
def format_relativedelta(t0, t1):
    if t1 >= t0:
        rd = relativedelta(t1, t0)
        sign = "+"
    else:
        rd = relativedelta(t0, t1)
        sign = "-"

    parts = []
    if rd.years:
        parts.append(f"{rd.years}y")
    if rd.months:
        parts.append(f"{rd.months}mo")
    if rd.days:
        parts.append(f"{rd.days}d")
    if rd.hours:
        parts.append(f"{rd.hours}h")
    if rd.minutes:
        parts.append(f"{rd.minutes}m")

    return f"{sign}{' '.join(parts) if parts else '0d'}"

# -----------------------
# Main loop
# -----------------------

all_valid_pairs = []
contrasts_different = 0

for pid, bdmap_ids in patient_to_bdmap.items():
    for ct0_bdmap, ct1_bdmap in permutations(bdmap_ids, 2):

        if ct0_bdmap not in bdmap_to_contrast or ct1_bdmap not in bdmap_to_contrast:
            continue

        if bdmap_to_contrast[ct0_bdmap] == bdmap_to_contrast[ct1_bdmap]:

            t0_unix = bdmap_to_time[ct0_bdmap]
            t1_unix = bdmap_to_time[ct1_bdmap]

            t0_dt = bdmap_to_datetime[ct0_bdmap]
            t1_dt = bdmap_to_datetime[ct1_bdmap]

            all_valid_pairs.append({
                "ct0_bdmap": ct0_bdmap,
                "ct1_bdmap": ct1_bdmap,
                "ct0_t": t0_unix,
                "ct1_t": t1_unix,
                "time_delta_readable": format_relativedelta(t0_dt, t1_dt)
            })

        else:
            contrasts_different += 1

In [15]:
all_datapoint_pairs = pd.DataFrame(all_valid_pairs)

In [16]:
all_datapoint_pairs["unix_delta"] = all_datapoint_pairs["ct1_t"] - all_datapoint_pairs["ct0_t"]

In [17]:
bdmaps_with_tumor_mask = [segmentation_path.split("/")[-3] for segmentation_path in paths_with_tumor_mask]
organs = ["_".join(segmentation_path.split("/")[-1].split("_")[:-1]) for segmentation_path in paths_with_tumor_mask]

In [18]:
unix_delta = all_datapoint_pairs["unix_delta"]
all_datapoint_pairs["normalized_time_delta"] = (unix_delta - unix_delta.mean()) / unix_delta.std()

In [20]:
all_datapoint_pairs.to_csv("UCSF_all_datapairs.csv",index=False)

In [19]:
datapoints_with_masks = all_datapoint_pairs[all_datapoint_pairs["ct0_bdmap"].isin(bdmaps_with_tumor_mask) 
                    & all_datapoint_pairs["ct1_bdmap"].isin(bdmaps_with_tumor_mask)]

In [20]:
from collections import defaultdict

bdmap_organ_mapping = defaultdict(set)

for bdmap_id, organ in zip(bdmaps_with_tumor_mask, organs):
    bdmap_organ_mapping[bdmap_id].add(organ)

In [21]:
datapoints_with_masks["shared_organs"] = [
    bdmap_organ_mapping.get(b0, set()) & bdmap_organ_mapping.get(b1, set())
    for b0, b1 in datapoints_with_masks[["ct0_bdmap", "ct1_bdmap"]].itertuples(index=False)
]

/local/rpinise1/3997157/ipykernel_534435/3193126487.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datapoints_with_masks["shared_organs"] = [


In [22]:
datapoints_with_masks

,ct0_bdmap,ct1_bdmap,ct0_t,ct1_t,time_delta_readable,unix_delta,normalized_time_delta,shared_organs
4354,BDMAP_00062056,BDMAP_00062404,1213364940,1666365600,+14y 4mo 8d 1h 31m,453000660,4.606775,{duodenum}
4355,BDMAP_00062056,BDMAP_00062966,1213364940,1205491620,-2mo 30d 3h 2m,-7873320,-0.080067,{duodenum}
4367,BDMAP_00062404,BDMAP_00062056,1666365600,1213364940,-14y 4mo 8d 1h 31m,-453000660,-4.606775,{duodenum}
4370,BDMAP_00062404,BDMAP_00062966,1666365600,1205491620,-14y 7mo 7d 4h 33m,-460873980,-4.686843,{duodenum}
4372,BDMAP_00062966,BDMAP_00062056,1205491620,1213364940,+2mo 30d 3h 2m,7873320,0.080067,{duodenum}
...,...,...,...,...,...,...,...,...
207317,BDMAP_00183688,BDMAP_00179328,1325376000,1330041600,+1mo 23d,4665600,0.047447,{stomach}
207318,BDMAP_00183688,BDMAP_00184636,1325376000,1349049600,+9mo,23673600,0.240748,{stomach}
207320,BDMAP_00184636,BDMAP_00173865,1349049600,1326672000,-8mo 15d,-22377600,-0.227568,{stomach}
207321,BDMAP_00184636,BDMAP_00179328,1349049600,1330041600,-7mo 7d,-19008000,-0.193301,{stomach}


In [23]:
expanded_datapoints = datapoints_with_masks.explode("shared_organs")

In [24]:
expanded_datapoints["shared_organs"].unique()

array(['duodenum', 'prostate', 'stomach', 'esophagus', 'spleen',
       'adrenal', 'gallbladder', 'bladder', 'uterus', 'colon', 'breast',
       nan], dtype=object)

In [26]:
expanded_datapoints["organ"] = expanded_datapoints["shared_organs"]

In [27]:
expanded_datapoints.drop("shared_organs", inplace=True, axis=1)

In [28]:
expanded_datapoints.dtypes

ct0_bdmap                 object
ct1_bdmap                 object
ct0_t                      int64
ct1_t                      int64
time_delta_readable       object
unix_delta                 int64
normalized_time_delta    float64
organ                     object
dtype: object

In [228]:
expanded_datapoints.to_csv("Datapairs_with_masks.csv",index=False)